In [1]:
import pandas as pd 
import numpy as np
import matplotlib.pyplot as plt
from transformers import (
    BertTokenizer,
    BertForMaskedLM,
    BertConfig,
    LineByLineTextDataset,
    DataCollatorForLanguageModeling,
    Trainer,
    TrainingArguments
)
import torch

c:\Users\ritik\anaconda3\envs\datascience310\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [ ]:
from transformers import BertModel

model = BertModel.from_pretrained("C:\\GitHub\\Multi-Model-Bias-Detection-and-Debiasing-the-News\\EDA\\Model_config")

In [ ]:
model_TEXT=model.load_state_dict(torch.load("bert_finetuned.pt"))
model_TEXT.eval()

In [ ]:
model_MultiModal=model.load_state_dict(torch.load("bert_finetuned.pt"))
model_MultiModal.eval()

In [2]:
import torch
from transformers import AutoTokenizer
from PIL import Image
import requests
from torchvision import transforms

In [3]:
text_tokenizer = AutoTokenizer.from_pretrained("bert-base-uncased")
image_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
])

In [ ]:
from torch.utils.data import Dataset, DataLoader
from PIL import Image

class DATA(Dataset):
    def __init__(self, dataframe, text_tokenizer, image_transform):
        self.df = dataframe
        self.text_tokenizer = text_tokenizer
        self.image_transform = image_transform

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        text = row['article_text']
        image_path = row['image_path']

        # tokenize text
        text_inputs = self.text_tokenizer(
            text, padding='max_length', truncation=True, max_length=512 , return_tensors='pt'
        )
        # transform image
        image = Image.open(image_path).convert('RGB')
        image_input = self.image_transform(image)

        #label = torch.tensor(row['MultiModal_Label'])
        
        input_ids= text_inputs['input_ids'][0]
        attention_mask= text_inputs['attention_mask']
        pixel_values = image_input
        labels =  int(row['MultiModal_Label'])
        return {
            'input_ids': torch.tensor(input_ids, dtype=torch.long),
            'attention_mask': torch.tensor(attention_mask, dtype=torch.long),
            'pixel_values': image_input,
            'labels': torch.tensor(labels, dtype=torch.long)
        }

# Then create dataset and loader

test_loader_dataset = DATA(X_test, text_tokenizer, image_transform)

training_loader = DataLoader(train_loader_dataset, batch_size=4, shuffle=True)
val_loader = DataLoader(test_loader_dataset, batch_size=4, shuffle=True)

In [ ]:
def valid_MultiModal(model, testing_loader):
    model.eval()
    n_correct = 0; n_wrong = 0; total = 0; tr_loss=0; nb_tr_steps=0; nb_tr_examples=0 ; True_pos=0; False_pos=0; True_neg=0; False_neg=0; n_True_pos=0;n_False_pos=0;n_True_neg=0;n_False_neg=0

    with torch.no_grad():
        for _, data in tqdm(enumerate(testing_loader, 0)):
            ids = data['input_ids'].to(device, dtype = torch.long)
            mask = data['attention_mask'].to(device, dtype = torch.long)
            pixel_values = data['pixel_values'].to(device, dtype = torch.float)
            targets = data['labels'].unsqueeze(1).to(device, dtype = torch.float)

            outputs = model(input_ids=ids,
            attention_mask=mask,
            pixel_values=pixel_values)

            loss = loss_function(outputs, targets)
            tr_loss += loss.item()
            preds = torch.sigmoid(outputs) > 0.5        
            n_correct += calcuate_accuracy(preds, targets.bool())

            n_True_pos,n_False_pos,n_True_neg,n_False_neg = calculate_truth_value(preds, targets)
            True_pos += n_True_pos
            False_pos += n_False_pos
            True_neg += n_True_neg
            False_neg += n_False_neg


            nb_tr_steps += 1
            nb_tr_examples+=targets.size(0)
            
            if _%5000==0:
                loss_step = tr_loss/nb_tr_steps
                accu_step = (n_correct*100)/nb_tr_examples

        labels = ["Biased", "Not Biased"]  

    for i in range(len(preds)):
        predicted_class = int(preds[i].item())  
        predicted_label = labels[predicted_class]
        predicted_prob = torch.sigmoid(outputs[i]).item()
        print(f"Prediction: {predicted_label}, Probability: {predicted_prob:.4f}")

    epoch_loss = tr_loss/nb_tr_steps
    epoch_accu = (n_correct*100)/nb_tr_examples
    print(f"Validation Loss Epoch: {epoch_loss}")
    print(f"Validation Accuracy Epoch: {epoch_accu}")
    
    return epoch_accu,True_pos,False_pos,True_neg,False_neg
acc,n_True_pos,n_False_pos,n_True_neg,n_False_neg = valid_MultiModal(model, val_loader)

In [ ]:
def Combined_Model(alpha):
    acc,n_True_pos,n_False_pos,n_True_neg,n_False_neg = valid(model, val_loader)
